# Attention and Transformers

**IMSE 774 · Neural Networks · Week 14 · Session 27 · November 24, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/11-transformers.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

# One hue for magnitudes, two hues around a neutral midpoint for signed values.
SEQ = LinearSegmentedColormap.from_list("seq", ["#ffffff", NDSU_GREEN])
DIV = LinearSegmentedColormap.from_list("div", [STEEL, "#f3f3f3", ACCENT])

## Embedding words


`fig-toy-embedding`


In [ ]:
people = {"king": ("he", "royal", "old"),      "queen": ("she", "royal", "old"),
          "prince": ("he", "royal", "young"),  "princess": ("she", "royal", "young"),
          "man": ("he", "common", "old"),      "woman": ("she", "common", "old"),
          "boy": ("he", "common", "young"),    "girl": ("she", "common", "young")}
context = {"royal": ["crown", "throne", "palace"], "common": ["village", "farm", "market"],
           "old": ["wise", "grey"],                 "young": ["small", "plays"]}

# 300 four-word sentences per person: the word, a pronoun, a place word, an age word.
rng = np.random.default_rng(774)
sentences = [[w, pron, rng.choice(context[rank]), rng.choice(context[age])]
             for w, (pron, rank, age) in people.items() for _ in range(300)]

words = list(people)
context_words = sorted({t for s in sentences for t in s[1:]})
counts = np.array([[sum(c in s for s in sentences if s[0] == w) for c in context_words]
                   for w in words], dtype=float)

# Positive pointwise mutual information, then keep the three strongest directions.
P = counts / counts.sum()
ppmi = np.maximum(np.log(np.maximum(P, 1e-12) / (P.sum(1, keepdims=True) * P.sum(0, keepdims=True))), 0)
U, S, _ = np.linalg.svd(ppmi - ppmi.mean(0))
E = U[:, :3] * S[:3]                      # one 3-number embedding per word
emb = dict(zip(words, E))

# Name each coordinate by the attribute it separates (done after the fact, by us).
attrs = {"gender": [p[0] == "she" for p in people.values()],
         "rank":   [p[1] == "royal" for p in people.values()],
         "age":    [p[2] == "old" for p in people.values()]}
name_of = {}
for name, flags in attrs.items():
    f = np.array(flags, dtype=float)
    name_of[int(np.argmax([abs(np.corrcoef(f, E[:, j])[0, 1]) for j in range(3)]))] = name
col = {v: k for k, v in name_of.items()}

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7), gridspec_kw={"width_ratios": [1, 1.3]})
ax = axes[0]
ax.imshow(E, cmap=DIV, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(3), [name_of.get(j, f"coordinate {j+1}") for j in range(3)])
ax.set_yticks(range(len(words)), words)
for i in range(len(words)):
    for j in range(3):
        ax.text(j, i, f"{E[i, j]:+.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if abs(E[i, j]) > 0.6 else "black")
ax.grid(False)
ax.set_title("Learned coordinates")

ax = axes[1]
gx, ry = col["gender"], col["rank"]
adults = ["king", "queen", "man", "woman"]   # the younger four differ only in the age coordinate, not drawn here
ax.scatter([emb[w][gx] for w in adults], [emb[w][ry] for w in adults], s=42, color=NDSU_GREEN, zorder=3)
for w in adults:
    ax.annotate(w, emb[w][[gx, ry]], xytext=(9, -3), textcoords="offset points", fontsize=10)
for a, b in [("man", "king"), ("woman", "queen")]:
    ax.annotate("", xy=emb[b][[gx, ry]], xytext=emb[a][[gx, ry]],
                arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1.8))
ax.set_xlabel("gender coordinate"); ax.set_ylabel("rank coordinate")
ax.set_title("Same arrow, two starting points")
ax.margins(0.22)
plt.tight_layout()
plt.show()

`word-arithmetic`


In [ ]:
def nearest(v, exclude):
    candidates = [w for w in words if w not in exclude]
    sims = [v @ emb[w] / (np.linalg.norm(v) * np.linalg.norm(emb[w])) for w in candidates]
    order = np.argsort(sims)[::-1]
    return [(candidates[i], sims[i]) for i in order[:2]]

print(f"{'expression':28s}  nearest word (cosine)   runner-up")
for a, b, c in [("king", "man", "woman"), ("prince", "boy", "girl"),
                ("king", "prince", "princess"), ("woman", "queen", "king")]:
    (w1, s1), (w2, s2) = nearest(emb[a] - emb[b] + emb[c], exclude=(a, b, c))
    print(f"{a + ' - ' + b + ' + ' + c:28s}  {w1:9s} ({s1:+.2f})       {w2} ({s2:+.2f})")

## The paint-store analogy


`fig-paint`


In [ ]:
cans = ["Sunny Yellow", "Purple Lollipop", "Green Monster", "Orange Crush", "Lunch with Teal", "Mellow Blue"]
paint = np.array([[1.00, 0.84, 0.00], [0.58, 0.25, 0.75], [0.20, 0.65, 0.25],
                  [1.00, 0.50, 0.05], [0.00, 0.60, 0.60], [0.35, 0.50, 0.85]])   # values (RGB)
request = "light yellow with a bit of dark orange"

vocab = sorted(set(request.split()) | {w.lower() for c in cans for w in c.split()})
colour_words = {"yellow", "orange", "purple", "green", "teal", "blue"}
# Query: the request's words, colour words counting double the others.
q = np.array([(1.0 if w in colour_words else 0.5) if w in request.split() else 0.0 for w in vocab])
# Keys: which words appear in each can's name.
K = np.array([[1.0 if w in c.lower().split() else 0.0 for w in vocab] for c in cans])

scores  = K @ q                                         # one dot product per can
weights = np.exp(3 * scores) / np.exp(3 * scores).sum() # softmax (sharpness 3)
mix     = weights @ paint                               # weighted average of the values

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.0), gridspec_kw={"width_ratios": [4, 1]})
ax = axes[0]
ax.barh(range(6), weights, color=paint, edgecolor="#333333", linewidth=0.8)
ax.set_yticks(range(6), cans); ax.invert_yaxis()
for i, (s, w) in enumerate(zip(scores, weights)):
    ax.text(w + 0.01, i, f"{w:.1%}   (score {s:.1f})", va="center", fontsize=9)
ax.set_xlim(0, 0.62); ax.set_xlabel("attention weight")
ax.set_title(f'Request: "{request}"'); ax.grid(axis="y", visible=False)
ax = axes[1]
ax.imshow(mix.reshape(1, 1, 3)); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.set_title("Mixture")
plt.tight_layout()
plt.show()

## Self-attention


`attention-from-scratch`


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.q = nn.Linear(D, D, bias=False)    # input -> query
        self.k = nn.Linear(D, D, bias=False)    # input -> key
        self.v = nn.Linear(D, D, bias=False)    # input -> value

    def forward(self, X, causal=False):                  # X: (N, D), one word per row
        Q, K, V = self.q(X), self.k(X), self.v(X)        # step 1: queries, keys, values
        S = Q @ K.T / math.sqrt(X.shape[-1])              # step 2: score every query against every key
        if causal:                                        # (masking, used later in this note)
            S = S.masked_fill(torch.ones_like(S).triu(1).bool(), float("-inf"))
        A = F.softmax(S, dim=-1)                          # each row of A sums to 1
        return A @ V, A                                   # steps 3-4: scale the values and add them up

torch.manual_seed(774)
X = torch.randn(5, 16)                 # five "words", each a vector of 16 numbers
layer = SelfAttention(16)
out, A = layer(X)

reference = F.scaled_dot_product_attention(layer.q(X), layer.k(X), layer.v(X))
print("output shape:           ", tuple(out.shape))
print("row sums of A:          ", [round(s, 4) for s in A.sum(-1).tolist()])
print("largest gap from PyTorch:", f"{(out - reference).abs().max().item():.1e}")

## Positional encoding


`fig-posenc`


In [ ]:
def positional_embedding(N, D):
    p = torch.arange(N)[:, None]
    i = torch.arange(0, D, 2)[None, :]
    angle = p / 10000 ** (i / D)
    P = torch.zeros(N, D)
    P[:, 0::2], P[:, 1::2] = torch.sin(angle), torch.cos(angle)
    return P

P = positional_embedding(50, 128)
fig, ax = plt.subplots(figsize=(8.4, 3.3))
im = ax.imshow(P, cmap=DIV, vmin=-1, vmax=1, aspect="auto", interpolation="nearest")
ax.set_xlabel("element of the vector"); ax.set_ylabel("position in the sentence")
ax.grid(False)
fig.colorbar(im, ax=ax, pad=0.02, label="value added")
plt.tight_layout()
plt.show()

`permutation-test`


In [ ]:
perm = torch.tensor([3, 0, 4, 1, 2])            # a fixed reordering of the five words

shuffled, _ = layer(X[perm])
print(f"no positional embedding:   largest change in any word's output = {(shuffled - out[perm]).abs().max().item():.1e}")

P5 = positional_embedding(5, 16)
with_pos, _     = layer(X + P5)
with_pos_shuf, _ = layer(X[perm] + P5)
print(f"with positional embedding: largest change in any word's output = {(with_pos_shuf - with_pos[perm]).abs().max().item():.2f}")

## Masking


`fig-mask`


In [ ]:
tokens  = ["[START]", "My", "dog", "loves", "taking", "long"]
targets = ["My", "dog", "loves", "taking", "long", "walks"]
allowed = np.tril(np.ones((6, 6)))

fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.imshow(allowed, cmap=SEQ, vmin=0, vmax=1.6)
for i in range(6):
    for j in range(6):
        if j > i:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="////",
                                       edgecolor="#9a9a9a", linewidth=0))
ax.set_xticks(range(6), tokens, rotation=30, ha="right")
ax.set_yticks(range(6), [f"predict “{t}”" for t in targets])
ax.set_xlabel("word in the decoder's input")
ax.set_xticks(np.arange(-0.5, 6), minor=True); ax.set_yticks(np.arange(-0.5, 6), minor=True)
ax.grid(False); ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", length=0)
plt.tight_layout()
plt.show()

`mask-check`


In [ ]:
masked, _ = layer(X, causal=True)
X_changed = X.clone()
X_changed[4] = torch.randn(16)                 # replace the last word with a different one
masked_changed, _ = layer(X_changed, causal=True)
unmasked_changed, _ = layer(X_changed)

print(f"with the mask:    outputs for words 1-4 change by at most {(masked_changed[:4] - masked[:4]).abs().max().item():.1e}")
print(f"without the mask: outputs for words 1-4 change by at most {(unmasked_changed[:4] - out[:4]).abs().max().item():.2f}")

## Transformers in action


`tiny-transformer`


In [ ]:
class Block(nn.Module):
    """One encoder block: self-attention and a feed-forward layer, each with skip connection + layer norm."""
    def __init__(self, D, H):
        super().__init__()
        self.attn = nn.MultiheadAttention(D, H, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(D, 4 * D), nn.ReLU(), nn.Linear(4 * D, D))
        self.norm1, self.norm2 = nn.LayerNorm(D), nn.LayerNorm(D)

    def forward(self, x):                       # x: (batch, N, D)
        a, weights = self.attn(x, x, x)         # queries, keys, values all from x
        x = self.norm1(x + a)
        return self.norm2(x + self.ff(x)), weights

class TinyTransformer(nn.Module):
    def __init__(self, vocab=10, N=8, D=32, H=1, use_position=True):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, D), nn.Embedding(N, D)
        self.block, self.out = Block(D, H), nn.Linear(D, vocab)
        self.use_position = use_position

    def forward(self, idx):                     # idx: (batch, N) integers
        x = self.tok(idx)                       # word embedding
        if self.use_position:
            x = x + self.pos(torch.arange(idx.shape[1]))   # positional embedding, added
        x, weights = self.block(x)
        return self.out(x), weights             # scores for each digit at each position

def train(model, steps=400):
    opt, losses = torch.optim.Adam(model.parameters(), lr=3e-3), []
    for _ in range(steps):
        x = torch.randint(0, 10, (128, 8))      # a batch of random digit strings
        y = x.flip(1)                           # the target is the same string reversed
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, 10), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

results = {}
for use_position in (True, False):
    torch.manual_seed(774)
    model = TinyTransformer(use_position=use_position)
    losses = train(model)
    x_test = torch.randint(0, 10, (2000, 8))
    logits, weights = model(x_test)
    accuracy = (logits.argmax(-1) == x_test.flip(1)).float().mean().item()
    results[use_position] = (losses, weights.mean(0).detach(), accuracy)
    print(f"positional embedding {'on ' if use_position else 'off'}: {accuracy:.1%} of digits correct on 2,000 new strings")

`fig-reverse`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.plot(results[True][0],  color=NDSU_GREEN, lw=1.8, label="with positional embedding")
ax.plot(results[False][0], color=ACCENT,     lw=1.8, label="without")
ax.set_xlabel("training step"); ax.set_ylabel("cross-entropy loss"); ax.legend()
ax = axes[1]
im = ax.imshow(results[True][1], cmap=SEQ, vmin=0, vmax=1)
ax.set_xlabel("input position read"); ax.set_ylabel("output position")
ax.set_xticks(range(8)); ax.set_yticks(range(8)); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="attention weight")
plt.tight_layout()
plt.show()

## GPT-2: decoder blocks only


`fig-temperature`


In [ ]:
candidates = ["roar", "bellow", "blast", "sigh", "daffodil"]
scores_z = np.array([4.0, 2.5, 2.0, 1.0, -2.0])

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for k, (T, colour) in enumerate([(0.5, NDSU_GREEN), (1.0, NDSU_GOLD), (2.0, ACCENT)]):
    p = np.exp(scores_z / T); p /= p.sum()
    ax.bar(np.arange(5) + (k - 1) * 0.27, p, width=0.25, color=colour, label=f"T = {T}")
ax.set_xticks(range(5), candidates); ax.set_ylabel("probability of being chosen")
ax.set_title("“The dragon approached and let out a mighty …”")
ax.grid(axis="x", visible=False); ax.legend(title="temperature")
plt.tight_layout()
plt.show()